In [1]:
import fitz  # PyMuPDF

def extract_text(path: str) -> str:
    doc = fitz.open(path) 
    pages = []
    for page in doc:
        page_text = page.get_text()
        pages.append(page_text)
    full_text = "\n".join(pages)
    return full_text

In [2]:
rules = extract_text("rules.pdf") + extract_text("rules2.pdf")

In [29]:
chunk_size = 10000
# Add 50% of chunk size to have overlapping chunks 
chunks = [rules[i: i+min(len(rules), chunk_size * 3 // 2)] for i in range(0, len(rules), chunk_size)]

In [18]:
from ollama import chat
import json

initial_messages = [
    {
        "role": "system",
        "content": """
        You will be generating synthetic data for supervised fine tuning. 
        The user will provide you with some rules about dungeons and dragons.
        You will provide questions and answers about the rules provided. If there is no relevant rules text, then don't generate any questions and answers.
        The question answer pair will be in the form {"question": string, "answer": string}. You will return them in an array in JSON format.
        """
    }
]
model = "gpt-oss:20b-cloud"

def strip_block_quotes(response):
    lines = response.split("\n")
    if "```" in lines[0] and "```" in lines[-1]:
        return '\n'.join(lines[1:-1])
    else:
        return response

def escape_backslashes(s: str) -> str:
    return s.replace("\\", "\\\\")

def generate_qa(chunk):
    response = chat(
        model=model,
        messages=[
            *initial_messages,
            {
                "role": "user",
                "content": chunk
            }
        ])
    raw = response["message"]["content"]
    stripped = strip_block_quotes(response["message"]["content"])

    content = escape_backslashes(stripped)
    pairs = json.loads(content)
    for pair in pairs:
        if not pair["question"] or not pair["answer"]:
            raise Exception(f"Unexpected format: {content}")
    return pairs

In [3]:
try: 
    print(len(text_to_pairs))
except NameError:
    text_to_pairs = dict() # Store text mapped to pairs in case some fail, we can retry and add to dictionary later

In [17]:
from IPython.display import clear_output

for chunk in chunks:
    opening = chunk[:100]
    clear_output(wait=True)
    print(f"{len(text_to_pairs)}/{len(chunks)}")
    if opening not in text_to_pairs:
        print(opening)
        try:
            text_to_pairs[opening] = generate_qa(chunk)
        except:
            print("Failed, skipping")
            pass
    else:
        print("Pairs already finished, skipping")

77/77
Pairs already finished, skipping


In [22]:
qa_pairs = [
    pair 
    for _, value in text_to_pairs.items()
    for pair in value
]
print(len(qa_pairs))
print(qa_pairs[0])
with open("synthetic-data-qa.json", "w") as f:
    json.dump(qa_pairs,f)

722
{'question': 'What does a monster with a climbing speed gain?', 'answer': 'A monster that has a climbing speed can use all or part of its movement to move on vertical surfaces. The monster doesn’t need to spend extra movement to climb.'}
